In [ ]:
   import pandas as pd
   print("pandas آماده‌ست:", pd.__version__)

In [ ]:
import os
import pandas as pd

candidates = pd.DataFrame({
    "gene": ["EGFR", "PDGFRA", "CDK4", "CDK6", "MET", "MDM2", "PIK3CA"],
    "target_class": [
        "receptor tyrosine kinase", "receptor tyrosine kinase",
        "cell-cycle kinase", "cell-cycle kinase",
        "receptor tyrosine kinase", "E3 ligase (p53 regulator)",
        "lipid kinase (PI3K pathway)",
    ],
})

candidates

In [ ]:
import pandas as pd

df = pd.read_csv("results/candidates.csv")

print("شکل جدول (ردیف, ستون):", df.shape)
print()
print("اسم ستون‌ها:")
for c in df.columns:
    print(" -", c)
print()
df

In [ ]:
cols = ["gene", "n_structures_under_3A", "alteration_frequency",
        "logrank_p", "dep_glioma_verdict"]

print(df[cols])
print()
print("مقدارهای یکتا در verdict:", df["dep_glioma_verdict"].unique())
print("تعداد خالی (NaN) در هر ستون:")
print(df[cols].isna().sum())

In [ ]:
def score_structure(n):
    if n >= 50:
        return 2
    elif n >= 10:
        return 1
    else:
        return 0

def score_alteration(pct):
    if pct >= 20:
        return 2
    elif pct >= 8:
        return 1
    else:
        return 0

def score_survival(p):
    # فقط p < 0.05 رو شرط گذاشتیم (جهت بقا رو چک نکردیم).
    # چون همه‌ی p ها بالای 0.1 هستن، برای همه 0 می‌شه.
    if p < 0.05:
        return 2
    else:
        return 0

def score_dependency(verdict):
    mapping = {"many": 2, "some": 1, "none": 0}
    return mapping[verdict]

df["s_structure"] = df["n_structures_under_3A"].apply(score_structure)
df["s_alteration"] = df["alteration_frequency"].apply(score_alteration)
df["s_survival"] = df["logrank_p"].apply(score_survival)
df["s_dependency"] = df["dep_glioma_verdict"].apply(score_dependency)

score_cols = ["s_structure", "s_alteration", "s_survival", "s_dependency"]
df["total_score"] = df[score_cols].sum(axis=1)

result = df[["gene"] + score_cols + ["total_score"]]
result = result.sort_values("total_score", ascending=False)
print(result)

In [ ]:
def score_threshold(value, high, low):
    if value >= high:
        return 2
    elif value >= low:
        return 1
    else:
        return 0

dep_map = {"many": 2, "some": 1, "none": 0}

def total_with(struct_hi, struct_lo, alt_hi, alt_lo):
    s1 = df["n_structures_under_3A"].apply(lambda v: score_threshold(v, struct_hi, struct_lo))
    s2 = df["alteration_frequency"].apply(lambda v: score_threshold(v, alt_hi, alt_lo))
    s3 = df["s_survival"]
    s4 = df["dependency_score"] if "dependency_score" in df else df["dep_glioma_verdict"].map(dep_map)
    return s1 + s2 + s3 + s4

settings = [
    ("اصلی",            50, 10, 20, 8),
    ("ساختار سخت‌تر",   100, 20, 20, 8),
    ("ساختار راحت‌تر",   30,  5, 20, 8),
    ("تغییر سخت‌تر",     50, 10, 30, 10),
    ("تغییر راحت‌تر",    50, 10, 15,  5),
    ("همه سخت‌تر",      100, 20, 30, 10),
    ("همه راحت‌تر",      30,  5, 15,  5),
]

table = pd.DataFrame({"gene": df["gene"]})
for name, sh, sl, ah, al in settings:
    table[name] = total_with(sh, sl, ah, al)

print(table.to_string(index=False))

In [ ]:
result.to_csv("results/scores.csv", index=False)
table.to_csv("results/sensitivity.csv", index=False)

In [ ]:
result.to_csv("results/scores.csv", index=False)
table.to_csv("results/sensitivity.csv", index=False)

In [ ]:
import requests

URL = "https://search.rcsb.org/rcsbsearch/v2/query"

def search_pdb(gene, with_ligand):
    nodes = [
        {"type": "terminal", "service": "text",
         "parameters": {"attribute": "rcsb_entity_source_organism.rcsb_gene_name.value",
                        "operator": "exact_match", "value": gene}},
        {"type": "terminal", "service": "text",
         "parameters": {"attribute": "rcsb_entity_source_organism.scientific_name",
                        "operator": "exact_match", "value": "Homo sapiens"}},
        {"type": "terminal", "service": "text",
         "parameters": {"attribute": "rcsb_entry_info.resolution_combined",
                        "operator": "less", "value": 3.0}},
    ]
    if with_ligand:
        nodes.append({"type": "terminal", "service": "text",
                      "parameters": {"attribute": "rcsb_entry_info.nonpolymer_entity_count",
                                     "operator": "greater", "value": 0}})
    query = {
        "query": {"type": "group", "logical_operator": "and", "nodes": nodes},
        "return_type": "entry",
        "request_options": {"paginate": {"start": 0, "rows": 3}},
    }
    r = requests.post(URL, json=query, timeout=60)
    if r.status_code == 204:      # 204 = no results found
        return 0, []
    r.raise_for_status()
    data = r.json()
    ids = [hit["identifier"] for hit in data["result_set"]]
    return data["total_count"], ids

n_good, n_ligand, examples = [], [], []
for gene in candidates["gene"]:
    print("checking", gene, "...")
    total, _ = search_pdb(gene, with_ligand=False)
    with_lig, ids = search_pdb(gene, with_ligand=True)
    n_good.append(total)
    n_ligand.append(with_lig)
    examples.append(", ".join(ids))
    print("  done:", total, "good structures,", with_lig, "with ligand")

candidates["n_structures_under_3A"] = n_good
candidates["n_with_ligand"] = n_ligand
candidates["example_pdb_ids"] = examples
candidates["structure_available"] = candidates["n_structures_under_3A"] > 0

candidates

In [ ]:
os.makedirs("results", exist_ok=True)
candidates.to_csv("results/candidates.csv", index=False)
print("saved!")

In [ ]:
candidates.to_excel("results/candidates.xlsx", index=False)
print("saved to excel!")

In [ ]:
alteration_pct = {
    "EGFR": 53,
    "PDGFRA": 16,
    "CDK4": 16,
    "CDK6": 3,
    "MET": 4,
    "MDM2": 9,
    "PIK3CA": 12,
}

candidates["alteration_frequency"] = candidates["gene"].map(alteration_pct)
candidates["alteration_source"] = "cBioPortal, GBM TCGA PanCancer Atlas, n=378"

candidates

In [ ]:
candidates.to_csv("results/candidates.csv", index=False)
candidates.to_excel("results/candidates.xlsx", index=False)
print("saved!")

In [ ]:
median_os_altered = {
    "EGFR": 14.93, "PDGFRA": None, "CDK4": None, "CDK6": None,
    "MET": None, "MDM2": None, "PIK3CA": None,
}
median_os_unaltered = {
    "EGFR": 12.99, "PDGFRA": None, "CDK4": None, "CDK6": None,
    "MET": None, "MDM2": None, "PIK3CA": None,
}
logrank_p = {
    "EGFR": 0.615, "PDGFRA": None, "CDK4": None, "CDK6": None,
    "MET": None, "MDM2": None, "PIK3CA": None,
}

candidates["median_os_altered"] = candidates["gene"].map(median_os_altered)
candidates["median_os_unaltered"] = candidates["gene"].map(median_os_unaltered)
candidates["os_diff_months"] = (
    candidates["median_os_altered"] - candidates["median_os_unaltered"]
)
candidates["logrank_p"] = candidates["gene"].map(logrank_p)
candidates["survival_source"] = "cBioPortal, GBM TCGA PanCancer Atlas, single-gene queries"

candidates

In [ ]:
import pandas as pd
import numpy as np

candidates = pd.read_csv("results/candidates.csv")
candidates.columns.tolist()

In [ ]:
median_os_altered = {
    "EGFR": 14.93, "PDGFRA": 13.12, "CDK4": 13.31, "CDK6": 12.23,
    "MET": 22.49, "MDM2": 14.20, "PIK3CA": 13.12,
}
median_os_unaltered = {
    "EGFR": 12.99, "PDGFRA": 14.53, "CDK4": 14.53, "CDK6": 14.53,
    "MET": 13.94, "MDM2": 14.53, "PIK3CA": 14.73,
}
logrank_p = {
    "EGFR": 0.615, "PDGFRA": 0.420, "CDK4": 0.497, "CDK6": 0.113,
    "MET": 0.126, "MDM2": 0.464, "PIK3CA": 0.104,
}

candidates["median_os_altered"] = candidates["gene"].map(median_os_altered)
candidates["median_os_unaltered"] = candidates["gene"].map(median_os_unaltered)
candidates["os_diff_months"] = (
    candidates["median_os_altered"] - candidates["median_os_unaltered"]
)
candidates["logrank_p"] = candidates["gene"].map(logrank_p)
candidates["survival_source"] = "cBioPortal, GBM TCGA PanCancer Atlas, single-gene queries"

candidates

In [ ]:
candidates.to_csv("results/candidates.csv", index=False)
candidates.to_excel("results/candidates.xlsx", index=False)
print("saved!")

In [ ]:
dep_glioma_n_left = {
    "EGFR": 2, "PDGFRA": 4, "CDK4": None, "CDK6": None,
    "MET": 3, "MDM2": None, "PIK3CA": 4,
}
dep_glioma_n_lines = {
    "EGFR": 106, "PDGFRA": 106, "CDK4": 106, "CDK6": 106,
    "MET": 106, "MDM2": 106, "PIK3CA": 106,
}
dep_glioma_verdict = {
    "EGFR": "none", "PDGFRA": "some", "CDK4": "some", "CDK6": "many",
    "MET": "some", "MDM2": "many", "PIK3CA": "some",
}

candidates["dep_glioma_n_left"] = candidates["gene"].map(dep_glioma_n_left)
candidates["dep_glioma_n_lines"] = candidates["gene"].map(dep_glioma_n_lines)
candidates["dep_glioma_fraction"] = (
    candidates["dep_glioma_n_left"] / candidates["dep_glioma_n_lines"]
)
candidates["dep_glioma_verdict"] = candidates["gene"].map(dep_glioma_verdict)
candidates["dep_glioma_source"] = (
    "DepMap CRISPR (Chronos), Adult-Type Diffuse Glioma lines; "
    "verdict by eye: none <=2 dots left of red line; some = more, but box edge right of line; "
    "many = box edge left of line"
)

candidates

In [ ]:
candidates.to_csv("results/candidates.csv", index=False)
candidates.to_excel("results/candidates.xlsx", index=False)
print(candidates.shape)
print(candidates.columns.tolist())

In [ ]:
import pandas as pd

model = pd.read_csv("data/Model.csv")
print(model.shape)
print(model.columns.tolist())

In [ ]:
import os
print("پوشه‌ی فعلی:", os.getcwd())
print("محتوای این پوشه:", os.listdir("."))
print()
if os.path.exists("data"):
    print("محتوای پوشه‌ی data:", os.listdir("data"))
else:
    print("پوشه‌ی data وجود نداره")

In [ ]:
import pandas as pd

model = pd.read_csv("data/Model.csv")
print(model.shape)
print(model.columns.tolist())

In [ ]:
print(model["OncotreeLineage"].value_counts().head(10))
print()

mask = model.apply(
    lambda row: "glioma" in " ".join(
        str(row[c]).lower() for c in ["OncotreeLineage", "OncotreePrimaryDisease", "OncotreeSubtype"]
    ),
    axis=1,
)
glioma_all = model[mask]
print("تعداد ردیف‌هایی که glioma توشون هست:", len(glioma_all))
print()
print(glioma_all["OncotreePrimaryDisease"].value_counts())
print()
print(glioma_all["OncotreeSubtype"].value_counts())

In [ ]:
crispr_head = pd.read_csv("data/CRISPRGeneEffect.csv", nrows=3)
print(crispr_head.shape)
print(crispr_head.columns[:5].tolist())
print([c for c in crispr_head.columns if c.startswith(("MDM2", "EGFR", "PDGFRA", "CDK4", "CDK6", "MET ", "PIK3CA", "TP53"))])

In [ ]:
genes = ["EGFR (1956)", "PDGFRA (5156)", "CDK4 (1019)", "CDK6 (1021)",
         "MET (4233)", "MDM2 (4193)", "PIK3CA (5290)"]

crispr = pd.read_csv("data/CRISPRGeneEffect.csv",
                     usecols=["Unnamed: 0"] + genes)
crispr = crispr.rename(columns={"Unnamed: 0": "ModelID"})

adult = model[model["OncotreePrimaryDisease"] == "Adult-Type Diffuse Glioma"]
dep = crispr.merge(adult[["ModelID", "CellLineName", "OncotreeSubtype"]],
                   on="ModelID", how="inner")

print("رده‌های گلیوما با داده‌ی CRISPR:", len(dep))
print()
rows = []
for g in genes:
    n_left = (dep[g] < -1).sum()
    rows.append((g.split(" ")[0], n_left, len(dep), round(n_left / len(dep), 3)))
print(pd.DataFrame(rows, columns=["gene", "n_below_-1", "n_lines", "fraction"]))

In [ ]:
dep.to_csv("results/dependency_glioma_counts_input.csv", index=False)
pd.DataFrame(rows, columns=["gene", "n_below_-1", "n_lines", "fraction"]).to_csv(
    "results/dependency_glioma_counts.csv", index=False)

In [ ]:
mut_head = pd.read_csv("data/OmicsSomaticMutations.csv", nrows=5)
print(mut_head.shape)
print(mut_head.columns.tolist())
mut_head

In [ ]:
use = ["ModelID", "HugoSymbol", "VariantInfo", "ProteinChange",
       "LikelyLoF", "Hotspot", "VepImpact"]

parts = []
for chunk in pd.read_csv("data/OmicsSomaticMutations.csv",
                         usecols=use, chunksize=200000):
    parts.append(chunk[chunk["HugoSymbol"] == "TP53"])

tp53 = pd.concat(parts, ignore_index=True)
print("تعداد ردیف جهش TP53:", len(tp53))
print("رده‌های یکتا با جهش TP53:", tp53["ModelID"].nunique())
print()
print(tp53["LikelyLoF"].value_counts(dropna=False))
print(tp53["Hotspot"].value_counts(dropna=False))
print(tp53["VepImpact"].value_counts(dropna=False))

In [ ]:
# 1) قاعده (قبل از دیدن نتیجه): LikelyLoF یا Hotspot یا VepImpact == HIGH
damaging = tp53[(tp53["LikelyLoF"]) | (tp53["Hotspot"]) | (tp53["VepImpact"] == "HIGH")]
mut_ids = set(damaging["ModelID"])
any_ids = set(tp53["ModelID"])

# 2) آیا هر رده اصلاً توالی‌یابی جهش دارد؟
seq_ids = set()
for chunk in pd.read_csv("data/OmicsSomaticMutations.csv",
                         usecols=["ModelID"], chunksize=500000):
    seq_ids.update(chunk["ModelID"].unique())

d = dep[["ModelID", "CellLineName", "OncotreeSubtype", "MDM2 (4193)"]].copy()
d["has_mutation_data"] = d["ModelID"].isin(seq_ids)
d["tp53_damaging"] = d["ModelID"].isin(mut_ids)
d["tp53_any"] = d["ModelID"].isin(any_ids)
d["mdm2_dependent"] = d["MDM2 (4193)"] < -1

print("رده‌های گلیوما با داده‌ی جهش:", d["has_mutation_data"].sum(), "از", len(d))
d = d[d["has_mutation_data"]]
print()
print(pd.crosstab(d["tp53_damaging"], d["mdm2_dependent"],
                  rownames=["TP53 damaging"], colnames=["MDM2 < -1"], margins=True))
print()
print(d.groupby("tp53_damaging")["MDM2 (4193)"].agg(["count", "median", "min", "max"]).round(2))

In [ ]:
print(pd.crosstab(d["tp53_any"], d["mdm2_dependent"],
                  rownames=["TP53 any mutation"], colnames=["MDM2 < -1"], margins=True))

In [ ]:
d.to_csv("results/mdm2_tp53_glioma.csv", index=False)

In [ ]:
cand = pd.read_csv("results/candidates.csv")

counts = pd.DataFrame(rows, columns=["gene", "n_below_-1", "n_lines", "fraction"])
n_left_map = dict(zip(counts["gene"], counts["n_below_-1"]))
n_lines_map = dict(zip(counts["gene"], counts["n_lines"]))

cand["dep_glioma_n_left"] = cand["gene"].map(n_left_map)
cand["dep_glioma_n_lines"] = cand["gene"].map(n_lines_map)
cand["dep_glioma_fraction"] = (cand["dep_glioma_n_left"] / cand["dep_glioma_n_lines"]).round(3)
cand["dep_glioma_source"] = (
    "Computed from DepMap 26Q3 CRISPRGeneEffect.csv: Adult-Type Diffuse Glioma lines (n=106) "
    "with Chronos < -1. Verdict labels (none/some/many) were assigned earlier by eye and "
    "are consistent with the computed fractions (many >= 25%)."
)

print(cand[["gene", "dep_glioma_n_left", "dep_glioma_n_lines",
            "dep_glioma_fraction", "dep_glioma_verdict"]])

In [ ]:
cand.to_csv("results/candidates.csv", index=False)
cand.to_excel("results/candidates.xlsx", index=False)
print("saved!")

In [ ]:
d.to_csv("results/mdm2_tp53_glioma.csv", index=False)